In [2]:
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path

PROCESSED_DIR = Path("./data/processed")


In [3]:
df = pd.read_csv(PROCESSED_DIR / "atp_matches.csv")
df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2006-580,Australian Open,Hard,128.0,G,2006-01-16,103,103908,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,48.0,713.0,81.0,492.0
1,2006-580,Australian Open,Hard,128.0,G,2006-01-16,107,103990,16.0,NaN,...,50.0,35.0,20.0,14.0,9.0,13.0,19.0,1490.0,22.0,1390.0
2,2006-580,Australian Open,Hard,128.0,G,2006-01-16,104,103206,25.0,NaN,...,82.0,50.0,12.0,17.0,17.0,26.0,28.0,1080.0,9.0,2175.0
3,2006-580,Australian Open,Hard,128.0,G,2006-01-16,105,102148,NaN,NaN,...,81.0,51.0,26.0,23.0,5.0,13.0,65.0,580.0,10.0,2050.0
4,2006-580,Australian Open,Hard,128.0,G,2006-01-16,106,103970,11.0,NaN,...,57.0,31.0,14.0,13.0,7.0,13.0,14.0,1635.0,21.0,1390.0


In [4]:
## first we only work on the first 5 years to get our initial b-scores
initial_df = df[(df["tourney_date"] >= "2006-01-01") & (df["tourney_date"] < "2011-01-01")].copy()

print(initial_df.shape)


(5800, 49)


In [5]:
t = pd.Timestamp("2010-12-31")
alpha_days = 365
initial_df["tourney_date"] = pd.to_datetime(initial_df["tourney_date"]) 

##following the logic of the paper on decay over time, can be tuned prob

initial_df["age_days"] = (t - initial_df["tourney_date"]).dt.days
initial_df["decay_weight"] = 1/ (1 + initial_df["age_days"] / alpha_days)

initial_df[["tourney_date", "age_days", "decay_weight"]].head()

,tourney_date,age_days,decay_weight
0,2006-01-16,1810,0.167816
1,2006-01-16,1810,0.167816
2,2006-01-16,1810,0.167816
3,2006-01-16,1810,0.167816
4,2006-01-16,1810,0.167816


In [6]:
##create the data frame for the edges to convert

edge_df = (initial_df
           .groupby(["loser_name", "winner_name"], as_index = False)["decay_weight"]
           .sum()
           .rename(columns={"loser_name": "source", "winner_name": "target", "decay_weight": "weight"}))

print(edge_df.head())
print(edge_df.shape)

             source             target    weight
0  Adrian Mannarino    Diego Junqueira  0.277567
1  Adrian Mannarino    Dmitry Tursunov  0.314384
2  Adrian Mannarino  Fernando Verdasco  1.087170
3  Adrian Mannarino       Marc Gicquel  0.395879
4  Adrian Mannarino      Tommy Robredo  0.384211
(4743, 3)


In [7]:
print(edge_df.columns)

Index(['source', 'target', 'weight'], dtype='str')


In [8]:
##now the actual graph

G0 = nx.from_pandas_edgelist(
    edge_df,
    source = "source",
    target = "target",
    edge_attr = "weight",
    create_using = nx.DiGraph()
)

print(G0.number_of_nodes())
print(G0.number_of_edges())

407
4743


In [9]:
## simple check if all players are in the graph, should be 

all_players = pd.unique(
    pd.concat([initial_df["winner_name"], initial_df["loser_name"]], ignore_index=True)
)

G0.add_nodes_from(all_players)

print(len(all_players))
print(G0.number_of_nodes())

407
407


In [10]:
bscore_dict = nx.eigenvector_centrality(
    G0,
    max_iter=1000,
    tol = 1e-9,
    weight = "weight"
)

bscore_df = (
    pd.DataFrame({
        "player": list(bscore_dict.keys()),
        "bscore_2010": list(bscore_dict.values())
    })
    .sort_values("bscore_2010", ascending = False)
    .reset_index(drop = True)
)

bscore_df.head()

,player,bscore_2010
0,Rafael Nadal,0.506170
1,Roger Federer,0.435596
2,Novak Djokovic,0.318824
3,Andy Murray,0.274846
4,Andy Roddick,0.197039


In [11]:
## scores should be normalized in euclidean norm 1, meaning if all went correctly we get norm = 1.0,
#  at least ca.

norm = np.sqrt((bscore_df["bscore_2010"] **2).sum())
print(norm)

1.0


In [12]:
matches_df = pd.read_csv(PROCESSED_DIR / "atp_matches.csv")

history_matches = matches_df[(matches_df["tourney_date"] >= "2006-01-01") & (matches_df["tourney_date"] < "2011-01-01")].copy()
history_matches["tourney_date"] = pd.to_datetime(history_matches["tourney_date"])

future_matches = matches_df[(matches_df["tourney_date"] >= "2011-01-01")].copy()
future_matches["tourney_date"] = pd.to_datetime(future_matches["tourney_date"])
print(history_matches.shape)
print(future_matches.shape)


(5800, 49)
(15423, 49)


In [13]:
def build_edge_df(match_history, t , alpha_days=365):
    df = match_history.copy()
    df["age_days"] = (t -df["tourney_date"]).dt.days
    df["decay_weight"] = 1/ (1 + df["age_days"] / alpha_days)
    edge_df = (df
           .groupby(["loser_name", "winner_name"], as_index = False)["decay_weight"]
           .sum()
           .rename(columns={"loser_name": "source", "winner_name": "target", "decay_weight": "weight"}))
    return edge_df

In [14]:
def build_graph(edge_df, all_players=None):
    G = nx.from_pandas_edgelist(
        edge_df,
        source = "source",
        target = "target",
        edge_attr = "weight",
        create_using = nx.DiGraph()
    )

    if all_players is not None:
        G.add_nodes_from(all_players)
        
    return G

In [15]:
def compute_bscore(G):
    bscore = nx.eigenvector_centrality(
        G,
        weight = "weight",
        max_iter=1000,
        tol=1e-9
    )
    return bscore

In [16]:
t0 = pd.Timestamp("2010-12-31")

all_players_0 = pd.unique(
    pd.concat(
        [history_matches["winner_name"], history_matches["loser_name"]],
        ignore_index=True

    )
)

edge_df_0 = build_edge_df(history_matches, t=t0, alpha_days=365)
G0 = build_graph(edge_df_0, all_players=all_players_0)
bscore_0 = compute_bscore(G0)

print(len(bscore_0))
print(list(bscore_0.items())[:5])


407
[('Adrian Mannarino', 0.00022386017249404836), ('Diego Junqueira', 4.5732781938795e-05), ('Dmitry Tursunov', 0.019151939319478062), ('Fernando Verdasco', 0.15449664117748088), ('Marc Gicquel', 0.012504213054316387)]


In [17]:
bscore_0_df = (
    pd.DataFrame({
        "player": list(bscore_0.keys()),
        "bscore": list(bscore_0.values())
    })
    .sort_values("bscore", ascending = False)
    .reset_index(drop = True)
)

default_bscore_0 = bscore_0_df["bscore"].quantile(0.25)

print("default_bscore_0:", default_bscore_0)
bscore_0_df.tail(10)

bscore_0_df.to_csv(PROCESSED_DIR / "bscore_0_general.csv", index = False)

default_bscore_0: 3.4789366760928304e-21


In [18]:
##cant update day by day because not how dataset is structured

rolling_dates = sorted(future_matches["tourney_date"].unique())

print(len(rolling_dates))
print(rolling_dates[:5])
print(rolling_dates[-5:])

224
[Timestamp('2011-01-17 00:00:00'), Timestamp('2011-03-23 00:00:00'), Timestamp('2011-04-10 00:00:00'), Timestamp('2011-04-18 00:00:00'), Timestamp('2011-04-24 00:00:00')]
[Timestamp('2024-07-01 00:00:00'), Timestamp('2024-07-15 00:00:00'), Timestamp('2024-08-05 00:00:00'), Timestamp('2024-08-12 00:00:00'), Timestamp('2024-10-28 00:00:00')]


In [19]:
sorted(matches_df["round"].dropna().unique())
round_order = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}

In [20]:
history_matches["round_order"] = history_matches["round"].map(round_order)

future_matches["round_order"] = future_matches["round"].map(round_order)

history_matches

rolling_round = (
    future_matches[["tourney_date", "tourney_id", "round_order"]]
    .drop_duplicates()
    .sort_values(["tourney_date","tourney_id", "round_order"])
    .reset_index(drop=True)
)
rolling_round.head(20)

,tourney_date,tourney_id,round_order
0,2011-01-17,2011-580,1
1,2011-01-17,2011-580,2
2,2011-01-17,2011-580,3
3,2011-01-17,2011-580,4
4,2011-01-17,2011-580,5
5,2011-01-17,2011-580,6
6,2011-01-17,2011-580,7
7,2011-03-23,2011-403,1
8,2011-03-23,2011-403,2
9,2011-03-23,2011-403,3


In [21]:
future_matches[future_matches["round_order"].isna()]["round"].unique()

<StringArray>
[]
Length: 0, dtype: str

In [22]:
current_history = history_matches.copy()
bscore_by_block = {}

enriched_blocks = []

for t_block, tourney_id, round_order in rolling_round.itertuples(index=False, name=None):
    block_matches = future_matches[
        (future_matches["tourney_id"] == tourney_id) &
        (future_matches["round_order"] == round_order)
        ].copy()
    
    t_block = block_matches["tourney_date"].iloc[0]

    all_players = pd.unique(
        pd.concat(
            [current_history["winner_name"], current_history["loser_name"]],
            ignore_index=True
        )
    )

    edge_df = build_edge_df(current_history, t=t_block, alpha_days=365)
    G = build_graph(edge_df, all_players=all_players)
    bscore_dict = compute_bscore(G)

    block_key = ("tourney_id", "round_order")
    bscore_by_block[block_key] = bscore_dict
    
    block_matches["winner_score_pre"] = block_matches["winner_name"].map(bscore_dict).fillna(default_bscore_0)
    block_matches["loser_score_pre"] = block_matches["loser_name"].map(bscore_dict).fillna(default_bscore_0)
    block_matches["bscore_diff"] = block_matches["winner_score_pre"] - block_matches["loser_score_pre"]

    enriched_blocks.append(block_matches) 

    current_history = pd.concat([current_history, block_matches], ignore_index=True)   

In [23]:
print(len(enriched_blocks))
print(enriched_blocks[0][["tourney_id", "round", "winner_name", "loser_name", "winner_score_pre", "loser_score_pre"]].head())

1451
     tourney_id round            winner_name     loser_name  winner_score_pre  \
5862   2011-580  R128          Roger Federer    Lukas Lacko          0.435933   
5864   2011-580  R128         Xavier Malisse  Pablo Andujar          0.027134   
5865   2011-580  R128       Marcos Baghdatis   Grega Zemlja          0.094813   
5866   2011-580  R128  Juan Martin del Potro      Dudi Sela          0.126760   
5867   2011-580  R128              Pere Riba   Carsten Ball          0.004381   

      loser_score_pre  
5862         0.006462  
5864         0.003305  
5865         0.002176  
5866         0.016866  
5867         0.001403  


In [24]:
print(enriched_blocks[6][["tourney_id", "round", "winner_name", "loser_name", "winner_score_pre", "loser_score_pre"]].head())

     tourney_id round     winner_name   loser_name  winner_score_pre  \
5832   2011-580     F  Novak Djokovic  Andy Murray           0.34492   

      loser_score_pre  
5832         0.278632  


In [25]:
final_matches_with_bscore = pd.concat(enriched_blocks, ignore_index=True)
final_matches_with_bscore.to_csv(PROCESSED_DIR / "atp_matches_with_bscore.csv", index=False)

In [26]:
final_matches_with_bscore[
    (final_matches_with_bscore["tourney_id"] == "2011-580") &
    (final_matches_with_bscore["winner_name"].str.contains("Federer", na=False))
][["tourney_id", "round", "winner_name", "winner_score_pre"]]

,tourney_id,round,winner_name,winner_score_pre
0,2011-580,R128,Roger Federer,0.435933
93,2011-580,R64,Roger Federer,0.433924
98,2011-580,R32,Roger Federer,0.434681
112,2011-580,R16,Roger Federer,0.434208
120,2011-580,QF,Roger Federer,0.434070
